<a href="https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My rule

A page is worth reviewing if it used to earn real search traffic, it hasn't been touched in
a while, and it isn't ranking well right now.

**Rule:** `stale (days_since_last_update >= 90)` AND `visible (impressions_90d >= 500)` AND
`weak_pos (avg_position >= 20 AND avg_position > 0)`. Score = the product of the three gates,
multiplied by `impressions_90d` — so among pages that pass all three gates, the ones with the
most search traffic at stake rank first.

**Reason codes** (one per row):
- `stale_weak_visible` — passed all three gates: stale, weakly positioned, and visible enough to matter
- `fresh_content` — updated recently (`days_since_last_update < 90`), so not a refresh candidate
- `low_visibility` — too little traffic (`impressions_90d < 500`) for a refresh to be worth it
- `ranking_ok` — already positioned well (`avg_position < 20`, and has position data)
- `no_position_data` — `avg_position == 0`, meaning no ranking data exists (flagged separately, not scored)

I moved the staleness cutoff from 180 to 90 days after checking the actual distribution:
`days_since_last_update` has a median of 20 and a 75th percentile of only 104, and the data
dictionary's own `181+` freshness tier holds just 174 of 30,000 rows — a 180-day cutoff nearly
emptied the rule. 90 days lines up with the dictionary's own `91-180` tier boundary and yields
9,345 stale pages, giving the rule something real to rank.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

def reason_code(row):
    if row["avg_position"] == 0:
        return "no_position_data"
    if row["days_since_last_update"] < 90:
        return "fresh_content"
    if row["impressions_90d"] < 500:
        return "low_visibility"
    if row["avg_position"] < 20:
        return "ranking_ok"
    return "stale_weak_visible"

df["reason_code"] = df.apply(reason_code, axis=1)
df["reason_code"].value_counts()

,count
reason_code,
fresh_content,19475
ranking_ok,4334
low_visibility,2745
stale_weak_visible,2241
no_position_data,1205


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Building the score

The score multiplies the three gates from Section 1 by `impressions_90d`, so pages that clear
all three thresholds are ranked by how much traffic is actually at stake. Pages that fail any
gate score 0 and sort to the bottom.

I also attach `is_declining_label` here (`trend_direction == "down"`) purely for evaluation
later — it is never an input to the score itself, since `trend_direction`/`trend_pct` are the
label source per the data dictionary.

In [7]:
# This cell is for CODE (numbers, a query, a check).
import os

stale = (df["days_since_last_update"] >= 90).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
weak_pos = ((df["avg_position"] >= 20) & (df["avg_position"] > 0)).astype(int)

df["score"] = stale * visible * weak_pos * df["impressions_90d"]

# label for evaluation only — never fed into the score above
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked[["content_id", "score", "reason_code", "is_declining_label"]].head(10)

,content_id,score,reason_code,is_declining_label
0,content_2dba2b1f9536,443434,stale_weak_visible,0
1,content_b28d1efd668f,286608,stale_weak_visible,0
2,content_813e88069237,233561,stale_weak_visible,1
3,content_b511d4bc4ad2,205915,stale_weak_visible,0
4,content_f02b48f88241,181514,stale_weak_visible,0
5,content_05e9b4cd9ccf,179002,stale_weak_visible,1
6,content_40fb6f005d61,151800,stale_weak_visible,1
7,content_8b36799b7e44,141400,stale_weak_visible,1
8,content_88d367c507a3,130932,stale_weak_visible,0
9,content_e752a4e03dd3,130892,stale_weak_visible,1


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Precision@20 and the top-20 hand review

Before reviewing individual rows, I check whether the rule beats random guessing.
`precision_at_k` measures how many of the top-K picks actually carry the declining label;
the base rate (`is_declining_label.mean()`) is what a random pick would score.


action: Review for refresh — this one's a genuine hit for the rule.
reason_code: stale_weak_visible (same as every top-20 row).
confidence note: High confidence. Unlike rows 1 and 5, this page is actually declining (trend_direction = down, −33.8%) on top of being stale and weakly positioned — so the rule's three static gates and the label agree here. Large volume (233k impressions) at a 0.06% CTR (very low even for position ~26) makes the traffic-at-stake argument strong too.
what would make it wrong: If the −33.8% drop is a temporary dip (seasonality, a SERP feature bump-out, a one-off algorithm fluctuation) rather than a structural content problem, a refresh wouldn't fix the real cause — and 104 days isn't that stale for a "keyword article," so the timing alone doesn't prove content decay caused the drop.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
p20 = precision_at_k(ranked["score"], ranked["is_declining_label"], 20)

print(f"Base rate (declining, whole dataset): {base_rate:.3f}")
print(f"Precision@20: {p20:.3f}")

# Display the top 20 rows the hand review below is based on
cols = ["content_id", "score", "impressions_90d", "avg_position",
        "days_since_last_update", "ctr", "trend_direction", "trend_pct",
        "is_declining_label"]
ranked[cols].head(20)

Base rate (declining, whole dataset): 0.542
Precision@20: 0.550


,content_id,score,impressions_90d,avg_position,days_since_last_update,ctr,trend_direction,trend_pct,is_declining_label
0,content_2dba2b1f9536,443434,443434,27.9,104,0.21,stable,1.4,0
1,content_b28d1efd668f,286608,286608,26.2,104,0.06,stable,-17.2,0
2,content_813e88069237,233561,233561,26.2,104,0.06,down,-33.8,1
3,content_b511d4bc4ad2,205915,205915,27.9,104,0.14,stable,-19.5,0
4,content_f02b48f88241,181514,181514,25.8,104,0.10,up,74.3,0
5,content_05e9b4cd9ccf,179002,179002,22.1,104,0.08,down,-42.1,1
6,content_40fb6f005d61,151800,151800,26.0,104,0.12,down,-37.6,1
7,content_8b36799b7e44,141400,141400,32.0,104,0.02,down,-62.7,1
8,content_88d367c507a3,130932,130932,40.1,104,0.04,stable,-13.7,0
9,content_e752a4e03dd3,130892,130892,23.9,104,0.01,down,-52.7,1


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

Three of the top 20 are flagged for refresh despite being pages that are *improving*, not
declining:

- **content_f02b48f88241** (row 5) — trend_direction "up", +74.3%. The rule sees a stale,
  weakly-positioned, high-traffic page and flags it; it has no way to see the page is already
  climbing without intervention. A refresh here would be wasted effort at best.
- **content_c9e444095b72** (row 16) — trend_direction "up", +64.7%. Same problem.
- **content_a2d6e73bc1eb** (row 18) — trend_direction "up", +24.1%. Same problem.

There's also a structural weak spot visible across the whole top 20: every single row has
`days_since_last_update == 104`. Among the ~2,241 pages that pass all three gates, the score
only differentiates by `impressions_90d` — it can't tell a page that's 91 days stale from one
that's 365 days stale, because staleness is a binary gate, not a ranked input. The rule is
really "sort stale-and-weak pages by traffic," not "find the most urgent refresh candidates."

## Leakage check

The score is built from `days_since_last_update`, `impressions_90d`, and `avg_position` only.
None of the label-source columns (`trend_direction`, `trend_pct`) or the 30-day windows they're
computed from (`impressions_last_30d`, `impressions_prev_30d`, and the

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Columns actually used to build the score (from Section 2)
score_inputs = {"days_since_last_update", "impressions_90d", "avg_position"}

# Columns that must NEVER appear in score_inputs
forbidden = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "provider_used", "model_used",
    "content_id", "client_id",
}

leaked = score_inputs & forbidden
print("Score inputs:", score_inputs)
print("Leaked columns found:", leaked if leaked else "None")
assert not leaked, "Leakage detected in score inputs!"

# Confirm the weak picks called out above
weak_picks = ranked[ranked["content_id"].isin([
    "content_f02b48f88241", "content_c9e444095b72", "content_a2d6e73bc1eb"
])][["content_id", "trend_direction", "trend_pct", "score"]]
weak_picks

Score inputs: {'avg_position', 'days_since_last_update', 'impressions_90d'}
Leaked columns found: None


,content_id,trend_direction,trend_pct,score
4,content_f02b48f88241,up,74.3,181514
15,content_c9e444095b72,up,64.7,110205
17,content_a2d6e73bc1eb,up,24.1,109568


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.